In [1]:
# ============================================================
# FUNCTION 8 - WEEK 13 FINAL-WEEK BAYESIAN OPTIMISATION
# Run from inside week13/
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)


# ------------------------------------------------------------
# 1. Load Week 13 cumulative data
# ------------------------------------------------------------

X = np.load(
    "function8/initial_inputs.npy"
)

Y = np.load(
    "function8/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

tree = cKDTree(X)

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 12 calibration check
# ------------------------------------------------------------
#
# Approximate Week 12 posterior prediction:
#
# mean ≈ 10.000546
# std  ≈ 0.004228
#
# Actual:
#
# 9.999147016163
#
# Error ≈ -0.33 sigma.
#
# This is excellent local calibration and supports strong
# final-week exploitation of the updated GP.
# ------------------------------------------------------------

week12_pred_mean = (
    10.000546
)

week12_pred_std = (
    0.004228
)

week12_actual = (
    9.999147016163
)

week12_error = (
    week12_actual
    - week12_pred_mean
)

week12_z_error = (
    week12_error
    / week12_pred_std
)

print("\n================================")
print("WEEK 12 CALIBRATION CHECK")
print("================================")

print(
    "Approx predicted mean:",
    week12_pred_mean
)

print(
    "Approx predicted std :",
    week12_pred_std
)

print(
    "Actual              :",
    week12_actual
)

print(
    "\nPrediction error:",
    week12_error
)

print(
    "\nError / predicted std:",
    week12_z_error
)


# ------------------------------------------------------------
# 3. Recent local evidence
# ------------------------------------------------------------

week11_x = np.array([
    0.107323,
    0.149041,
    0.129094,
    0.157441,
    0.815313,
    0.493175,
    0.201585,
    0.586041
])

week11_y = (
    9.9996456622444
)

week12_x = np.array([
    0.088567,
    0.167731,
    0.131357,
    0.163034,
    0.800372,
    0.500019,
    0.206649,
    0.588480
])

week12_y = (
    9.999147016163
)

print("\n================================")
print("RECENT LOCAL EVIDENCE")
print("================================")

print("\nWeek 11 / incumbent:")
print(
    week11_x,
    "->",
    week11_y
)

print("\nWeek 12:")
print(
    week12_x,
    "->",
    week12_y
)

print(
    "\nWeek 11 -> Week 12 displacement:"
)

print(
    week12_x
    - week11_x
)

print(
    "\nDistance Week 11 -> Week 12:"
)

print(
    np.linalg.norm(
        week12_x
        - week11_x
    )
)

print(
    "\nObjective difference:"
)

print(
    week12_y
    - week11_y
)


# ------------------------------------------------------------
# 4. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(8) * 0.4,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=15,
    random_state=42
)

gp.fit(
    X,
    Y
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0
    / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)

print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)

print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Final week:
# stay tightly inside the proven high-performing basin.
empirical_cap = min(
    nearest_distance,
    0.025
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 7. ARD-informed trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.06 * lengthscales,
    0.010,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 13 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)

print(
    trust_half_width
)

print(
    "\nLower:"
)

print(
    lower
)

print(
    "\nUpper:"
)

print(
    upper
)


# ------------------------------------------------------------
# 8. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

candidates = rng.uniform(
    lower,
    upper,
    size=(
        300000,
        8
    )
)

# Portal precision before duplicate filtering
candidates = np.round(
    candidates,
    6
)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "\nCandidates after portal rounding + duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 9. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 10. EI - diagnostic only
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI - DIAGNOSTIC ONLY")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "mean =",
    mu[ei_idx]
)

print(
    "std =",
    sigma[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 11. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"\nbeta={beta}"
    )

    print(
        "candidate =",
        candidates[idx]
    )

    print(
        "mean =",
        mu[idx]
    )

    print(
        "std =",
        sigma[idx]
    )

    print(
        "distance =",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:

        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 14. Axis-profile diagnostics
# ------------------------------------------------------------

print("\n================================")
print("AXIS PROFILE DIAGNOSTICS")
print("================================")

for j in range(8):

    values = np.linspace(
        lower[j],
        upper[j],
        35000
    )

    profile = np.tile(
        best_x,
        (
            len(values),
            1
        )
    )

    profile[:, j] = values

    profile = np.round(
        profile,
        6
    )

    profile_distance, _ = tree.query(
        profile,
        k=1
    )

    profile_valid = profile[
        profile_distance > 0.01
    ]

    if len(profile_valid) == 0:

        print(
            f"\nx{j+1}: no admissible points"
        )

        continue

    profile_mu, profile_sigma = gp.predict(
        profile_valid,
        return_std=True
    )

    idx = np.argmax(
        profile_mu
    )

    print(
        f"\nx{j+1} profile best:"
    )

    print(
        "candidate =",
        profile_valid[idx]
    )

    print(
        "mean =",
        profile_mu[idx]
    )

    print(
        "std =",
        profile_sigma[idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            profile_valid[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 15. Week 11 <-> Week 12 line profile
# ------------------------------------------------------------
#
# The two most recent points are both excellent.
#
# Search:
#
#   t = 0   -> Week 11 incumbent
#   t = 1   -> Week 12
#   t < 0   -> opposite side of incumbent
#   0<t<1   -> interpolation between Weeks 11 and 12
#
# This lets the GP decide whether the local optimum may lie
# between them or just beyond the incumbent in the opposite
# direction.
# ------------------------------------------------------------

direction = (
    week12_x
    - best_x
)

t_values = np.linspace(
    -0.8,
    1.2,
    100000
)

line_candidates = (
    best_x
    + t_values[:, None]
    * direction
)

valid_bounds = np.all(
    (
        line_candidates >= 0.0
    )
    &
    (
        line_candidates <= 1.0
    ),
    axis=1
)

line_candidates = line_candidates[
    valid_bounds
]

line_candidates = np.round(
    line_candidates,
    6
)

line_distance, _ = tree.query(
    line_candidates,
    k=1
)

line_valid = line_candidates[
    line_distance > 0.01
]

print("\n================================")
print("WEEK 11 <-> WEEK 12 LINE PROFILE")
print("================================")

if len(line_valid) > 0:

    line_mu, line_sigma = gp.predict(
        line_valid,
        return_std=True
    )

    line_idx = np.argmax(
        line_mu
    )

    print(
        "Best line candidate =",
        line_valid[line_idx]
    )

    print(
        "mean =",
        line_mu[line_idx]
    )

    print(
        "std =",
        line_sigma[line_idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            line_valid[line_idx]
            - best_x
        )
    )

else:

    print(
        "No admissible line candidates."
    )


# ------------------------------------------------------------
# 16. Exact midpoint diagnostic
# ------------------------------------------------------------

midpoint = (
    0.5
    * (
        week11_x
        + week12_x
    )
)

midpoint = np.round(
    midpoint,
    6
)

midpoint_dist, midpoint_idx = tree.query(
    midpoint.reshape(1, -1),
    k=1
)

print("\n================================")
print("WEEK 11 / WEEK 12 MIDPOINT")
print("================================")

print(
    "candidate =",
    midpoint
)

print(
    "nearest observed distance =",
    midpoint_dist[0]
)

print(
    "nearest observed point =",
    X[midpoint_idx[0]]
)

if midpoint_dist[0] > 0.01:

    midpoint_mu, midpoint_sigma = gp.predict(
        midpoint.reshape(1, -1),
        return_std=True
    )

    print(
        "mean =",
        midpoint_mu[0]
    )

    print(
        "std =",
        midpoint_sigma[0]
    )

else:

    print(
        "Midpoint is too close to an existing observation."
    )


# ------------------------------------------------------------
# 17. Opposite-side / mirror diagnostic
# ------------------------------------------------------------
#
# Week 12 was slightly worse than Week 11.
#
# Reflect the Week 12 displacement through the incumbent to
# test the opposite side of the local basin.
# ------------------------------------------------------------

mirror_candidate = (
    best_x
    - (
        week12_x
        - best_x
    )
)

mirror_candidate = np.clip(
    mirror_candidate,
    0.0,
    1.0
)

mirror_candidate = np.round(
    mirror_candidate,
    6
)

mirror_dist, mirror_idx = tree.query(
    mirror_candidate.reshape(1, -1),
    k=1
)

print("\n================================")
print("WEEK 12 MIRROR DIAGNOSTIC")
print("================================")

print(
    "candidate =",
    mirror_candidate
)

print(
    "nearest observed distance =",
    mirror_dist[0]
)

print(
    "nearest observed point =",
    X[mirror_idx[0]]
)

if mirror_dist[0] > 0.01:

    mirror_mu, mirror_sigma = gp.predict(
        mirror_candidate.reshape(1, -1),
        return_std=True
    )

    print(
        "mean =",
        mirror_mu[0]
    )

    print(
        "std =",
        mirror_sigma[0]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            mirror_candidate
            - best_x
        )
    )

else:

    print(
        "Mirror candidate is too close to an existing observation."
    )

DATA
X shape: (52, 8)
Y shape: (52,)

Current best:
[0.107323 0.149041 0.129094 0.157441 0.815313 0.493175 0.201585 0.586041] -> 9.9996456622444

Y range:
min = 5.5921933895401965
max = 9.9996456622444
std = 1.2263175877660981

WEEK 12 CALIBRATION CHECK
Approx predicted mean: 10.000546
Approx predicted std : 0.004228
Actual              : 9.999147016163

Prediction error: -0.0013989838369994345

Error / predicted std: -0.3308854865183147

RECENT LOCAL EVIDENCE

Week 11 / incumbent:
[0.107323 0.149041 0.129094 0.157441 0.815313 0.493175 0.201585 0.586041] -> 9.9996456622444

Week 12:
[0.088567 0.167731 0.131357 0.163034 0.800372 0.500019 0.206649 0.58848 ] -> 9.999147016163

Week 11 -> Week 12 displacement:
[-0.018756  0.01869   0.002263  0.005593 -0.014941  0.006844  0.005064
  0.002439]

Distance Week 11 -> Week 12:
0.032236207717409926

Objective difference:
-0.0004986460814002669


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co


GP FIT

Fitted kernel:
0.849**2 * Matern(length_scale=[1.29, 2, 1.01, 2, 2, 2, 1.37, 2], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[1.2880862  2.         1.01372036 2.         2.         2.
 1.36932618 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.15548377 0.10013825 0.19756582 0.10013825 0.10013825 0.10013825
 0.14625916 0.10013825]

EMPIRICAL LOCAL SCALE
Nearest observed point: 0.032236207717409926

Empirical cap: 0.025

WEEK 13 TRUST REGION
Centre:
[0.107323 0.149041 0.129094 0.157441 0.815313 0.493175 0.201585 0.586041]

Half-widths:
[0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025]

Lower:
[0.082323 0.124041 0.104094 0.132441 0.790313 0.468175 0.176585 0.561041]

Upper:
[0.132323 0.174041 0.154094 0.182441 0.840313 0.518175 0.226585 0.611041]

Candidates after portal rounding + duplicate filtering:
299994

PRIMARY EI - DIAGNOSTIC ONLY
candidate = [0.084632 0.127063 0.108497 0.134545 0.792164 0.474048 0.213003 0.609518]
mean = 9.999325094933866
std 

In [2]:
# ============================================================
# FINAL FUNCTION 8 - WEEK 13 SELECTION
# ============================================================
#
# FINAL CAPSTONE QUERY
#
# Evidence:
# - Week 11 remains the observed incumbent at 9.999645662.
# - Week 12 was extremely close at 9.999147016.
# - Week 12 calibration was strong:
#
#       prediction error ≈ -0.33 sigma
#
#   supporting trust in the local GP for the final query.
#
# - The updated highest-posterior-mean candidate predicts:
#
#       mean = 10.000729
#       std  = 0.005006
#
#   which is above the current observed best.
#
# - Low-beta UCB supports broadly the same direction.
# - Several axis profiles independently support components
#   of the same move, especially lower x1, x4, x6 and x8.
# - The Week11 <-> Week12 line gives a lower predicted mean,
#   while the Week12 mirror is clearly weaker.
# - EI / high-beta UCB move farther away and place more weight
#   on uncertainty, which has little value in the final round.
#
# Final-week strategy:
# exploit the well-calibrated local posterior and select the
# point with the highest predicted objective.
# ============================================================

week13_candidate = np.array([
    0.095001,
    0.139292,
    0.119306,
    0.147069,
    0.819451,
    0.469514,
    0.203887,
    0.575004
])

print("================================")
print("FUNCTION 8 - WEEK 13 FINAL")
print("================================")

print("\nCandidate:")
print(week13_candidate)


# ------------------------------------------------------------
# GP prediction
# ------------------------------------------------------------

final_mu, final_sigma = gp.predict(
    week13_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])


# ------------------------------------------------------------
# Predicted improvement over incumbent
# ------------------------------------------------------------

print("\nPredicted improvement over current best:")
print(
    final_mu[0] - best_y
)


# ------------------------------------------------------------
# Distance from incumbent
# ------------------------------------------------------------

distance_from_best = np.linalg.norm(
    week13_candidate - best_x
)

print("\nDistance from current best:")
print(distance_from_best)


# ------------------------------------------------------------
# Duplicate-rule verification
# ------------------------------------------------------------

nearest_dist, nearest_idx = tree.query(
    week13_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

print("\nNearest observed point:")
print(X[nearest_idx[0]])

print("\nDuplicate rule satisfied:")
print(
    nearest_dist[0] > 0.01
)


# ------------------------------------------------------------
# Portal format
# ------------------------------------------------------------

portal = "-".join(
    f"{x:.6f}"
    for x in week13_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 8 - WEEK 13 FINAL

Candidate:
[0.095001 0.139292 0.119306 0.147069 0.819451 0.469514 0.203887 0.575004]

Predicted mean:
10.000728761831326

Predicted std:
0.005005717519079728

Predicted improvement over current best:
0.0010830995869248738

Distance from current best:
0.03397555814111082

Nearest observed-point distance:
0.03397555814111082

Nearest observed point:
[0.107323 0.149041 0.129094 0.157441 0.815313 0.493175 0.201585 0.586041]

Duplicate rule satisfied:
True

Portal format:
0.095001-0.139292-0.119306-0.147069-0.819451-0.469514-0.203887-0.575004
